Hypothesis H-C: All objective sleep quality (OSQ) measures; total sleep time (TST), sleep onset latency (SOL), wake after sleep onset (WASO), and sleep efficiency (SE); are meaningful predictors of subjective sleep quality (SSQ).

**Please note that the analyses shown in this notebook were performed on synthetic data, and the results do not reflect the actual outcomes of the study.**

For more details, see the related preregistration at osf.io.

In [39]:
import pandas as pd
import numpy as np

In [40]:
strength = 30  # a slider [0, 100] that determines the amount of randomness in data;
               # 0: more random, less correlation; 100: less random: more correlation
rng = np.random.default_rng(9)  # to keep things reproducible

n = 1000

# Subjective Likert-scale sleep quality (1..5)
dataset_1 = pd.DataFrame({
    "Subj_likert": rng.integers(1, 6, size=n)})

def quality_from_likert(likert):
    return (likert - 1) / 4.0

# Generate objective sleep metrics with tunable correlation to subjective quality
def synthesize_metric(q, low_val, high_val, strength_0_100=0, positive=True, jitter=0.0):
    s = np.clip(strength_0_100 / 100.0, 0.0, 1.0)
    base_noise = rng.random(len(q))
    trend = q if positive else (1.0 - q)
    z = (1.0 - s) * base_noise + s * trend
    if jitter > 0:
        z = np.clip(z + rng.normal(0, jitter, size=len(z)), 0, 1)
    out = low_val + z * (high_val - low_val)
    return out

# Generating random values for objevtive metrics
target_q = quality_from_likert(dataset_1["Subj_likert"].values)
# TST: minutes, higher is better. [~300–600]
dataset_1["TST"] = synthesize_metric(target_q, low_val=300, high_val=600,
                                      strength_0_100=strength, positive=True, jitter=0.01).round().astype(int)

# SE: proportion, higher is better. [~0.60–0.98].
dataset_1["SE"] = np.round(
    synthesize_metric(target_q, low_val=0.50, high_val=1,
                      strength_0_100=strength, positive=True, jitter=0.01),3)

# SOL: minutes, lower is better. [~30–100].
dataset_1["SOL"] = synthesize_metric(target_q, low_val=30, high_val=100,
                                      strength_0_100=strength, positive=False, jitter=0.01).round().astype(int)

# WASO: minutes, lower is better.[~10–100].
dataset_1["WASO"] = synthesize_metric(target_q, low_val=10, high_val=100,
                                       strength_0_100=strength, positive=False, jitter=0.01).round().astype(int)

print("Mock data:\n")
print(dataset_1.iloc[0:10, :].to_string(index=False))

Mock data:

 Subj_likert  TST    SE  SOL  WASO
           3  388 0.583   67    28
           5  466 0.732   51    12
           5  485 0.725   39    31
           2  373 0.587   49    44
           1  354 0.759   71    57
           4  423 0.968   77    59
           4  550 0.733   77    64
           4  376 0.906   75    36
           4  498 0.630   66    58
           4  446 0.808   50    80


In [41]:
def summarize_clm_or(df, scale_note="Ordered-logit", outcome="SSQ (Likert)"):
    default_units = {
        "TST": "per 1 hour",
        "SE": "per 0.10 (10 pp)",
        "SOL": "per 10 min",
        "WASO": "per 10 min"}

    for _, r in df.iterrows():
        metric = str(r["metric"])
        or_val = float(r["OR"])
        lo = float(r["CI_low"])
        hi = float(r["CI_high"])
        unit = r.get("unit", default_units.get(metric, ""))

        # rules
        ci_excludes_1 = not (lo <= 1 <= hi)
        passes_thresh = (or_val >= 1.5) or (or_val <= 0.67)
        accepted = ci_excludes_1 and passes_thresh

        # labels
        thresh_txt = "OR≥1.5 or OR≤0.67 is met: " + ("yes" if passes_thresh else "no")
        ci_txt = "CI excludes 1: " + ("yes" if ci_excludes_1 else "no")
        verdict = "Predictor is meaningful: " + ("hypothesis is accepted" if accepted else "hypothesis is rejected")

        # sentence
        unit_txt = f" ({unit})" if unit else ""
        print(
            f"{scale_note} OR for {metric}{unit_txt} predicting {outcome} "
            f"was {or_val:.3f} (95% CI: {lo:.3f}–{hi:.3f}); \n\t{thresh_txt}; {ci_txt}; {verdict}.")

In [42]:
# Cumulative link (proportional-odds) model (CLM)

from statsmodels.miscmodels.ordinal_model import OrderedModel

X = (
    dataset_1[['TST','SE','SOL','WASO']].copy()
      .assign(
          TST=lambda d: d['TST']/60.0,         # hours
          SE=lambda d: d['SE']* 10.0,          # 10-percent
          SOL=lambda d: d['SOL']/10.0,         # per 10 min
          WASO=lambda d: d['WASO']/10.0        # per 10 min
      ))
y = dataset_1['Subj_likert'].astype(int)

model = OrderedModel(y, X, distr='logit')
result = model.fit(method='bfgs')
print(result.summary())


Optimization terminated successfully.
         Current function value: 1.243830
         Iterations: 25
         Function evaluations: 27
         Gradient evaluations: 27
                             OrderedModel Results                             
Dep. Variable:            Subj_likert   Log-Likelihood:                -1243.8
Model:                   OrderedModel   AIC:                             2504.
Method:            Maximum Likelihood   BIC:                             2543.
Date:                Sun, 16 Nov 2025                                         
Time:                        22:22:38                                         
No. Observations:                1000                                         
Df Residuals:                     992                                         
Df Model:                           4                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------

In [43]:
# Calculating odds ratios (OR):
b = result.params[X.columns]
se = result.bse[X.columns]
or_table = pd.DataFrame({
    'OR': np.exp(b),
    'CI_low': np.exp(b - 1.96*se),
    'CI_high': np.exp(b + 1.96*se),
    'p': result.pvalues[X.columns]})
or_table = or_table.rename_axis('metric').reset_index()
print(or_table)

  metric        OR    CI_low   CI_high             p
0    TST  2.145665  1.913860  2.405546  3.837615e-39
1     SE  1.978898  1.766302  2.217081  5.515680e-32
2    SOL  0.627126  0.579833  0.678276  1.942451e-31
3   WASO  0.707587  0.664524  0.753442  3.555004e-27


In [44]:
summarize_clm_or(or_table, scale_note="Ordered-logit", outcome="SSQ (Likert)")

Ordered-logit OR for TST (per 1 hour) predicting SSQ (Likert) was 2.146 (95% CI: 1.914–2.406); 
	OR≥1.5 or OR≤0.67 is met: yes; CI excludes 1: yes; Predictor is meaningful: hypothesis is accepted.
Ordered-logit OR for SE (per 0.10 (10 pp)) predicting SSQ (Likert) was 1.979 (95% CI: 1.766–2.217); 
	OR≥1.5 or OR≤0.67 is met: yes; CI excludes 1: yes; Predictor is meaningful: hypothesis is accepted.
Ordered-logit OR for SOL (per 10 min) predicting SSQ (Likert) was 0.627 (95% CI: 0.580–0.678); 
	OR≥1.5 or OR≤0.67 is met: yes; CI excludes 1: yes; Predictor is meaningful: hypothesis is accepted.
Ordered-logit OR for WASO (per 10 min) predicting SSQ (Likert) was 0.708 (95% CI: 0.665–0.753); 
	OR≥1.5 or OR≤0.67 is met: no; CI excludes 1: yes; Predictor is meaningful: hypothesis is rejected.


**############################################################**




In [45]:
# Create true random data

n = 5000
rng = np.random.default_rng(9)
dataset_1 = pd.DataFrame({
    "TST": rng.integers(300, 601, size=n),
    "SE": np.round(rng.random(n), 3),
    "SOL": rng.integers(10, 101, size=n),
    "WASO": rng.integers(10, 101, size=n),
    "Subj": rng.integers(1, 6, size=n)
})
